# CSC4093/DSC4213: Neural Networks and Deep Learning
## Programming Assignment 01 - LSTMs
### Personal Health Mention Classification from Tweets
---
**Task:** Classify tweets as Personal Health Mentions (1) or Non-Personal Health Mentions (0)  
**Models:** LSTM and Bidirectional LSTM (Bi-LSTM)

In [ ]:
# Import all libraries needed
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')
import warnings
warnings.filterwarnings('ignore')

from nltk.corpus import stopwords
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

nltk.download('stopwords')
english_stops = set(stopwords.words('english'))

print('All libraries imported successfully.')

## Step 1: Load and Preview the Dataset

In [ ]:
# Load the training and test datasets
train_data = pd.read_csv('phm_train.csv')
test_data  = pd.read_csv('phm_test.csv')

print('Training set shape:', train_data.shape)
print('Test set shape    :', test_data.shape)
print()
print('Training set preview:')
print(train_data.head())
print()
print('Label distribution in training set:')
print(train_data['label'].value_counts())
print()
print('Label distribution in test set:')
print(test_data['label'].value_counts())

## Step 2: Text Preprocessing

In [ ]:
def preprocess_tweets(data):
    x_data = data['tweet'].copy()
    y_data = data['label'].copy()

    # Remove URLs
    x_data = x_data.replace({'http\S+|www\S+': ''}, regex=True)
    # Remove user_mention placeholders
    x_data = x_data.replace({'user_mention': ''}, regex=True)
    # Remove HTML tags
    x_data = x_data.replace({'<.*?>': ''}, regex=True)
    # Remove non-alphabet characters
    x_data = x_data.replace({'[^A-Za-z]': ' '}, regex=True)
    # Tokenize, remove stop words, lowercase
    x_data = x_data.apply(lambda tweet: [w.lower() for w in tweet.split() if w.lower() not in english_stops and len(w) > 1])

    return x_data, y_data

x_train, y_train = preprocess_tweets(train_data)
x_test,  y_test  = preprocess_tweets(test_data)

print('Tweets (Training)')
print(x_train.head())
print()
print('Labels (Training)')
print(y_train.head())

## Step 3: Tokenization and Padding

In [ ]:
def get_max_length(x_data):
    tweet_lengths = [len(tweet) for tweet in x_data]
    return int(np.ceil(np.mean(tweet_lengths)))

# Fit tokenizer on training data only
token = Tokenizer(lower=False)
token.fit_on_texts(x_train)

x_train_seq = token.texts_to_sequences(x_train)
x_test_seq  = token.texts_to_sequences(x_test)

max_length  = get_max_length(x_train)
total_words = len(token.word_index) + 1  # +1 for padding index 0

x_train_pad = pad_sequences(x_train_seq, maxlen=max_length, padding='post', truncating='post')
x_test_pad  = pad_sequences(x_test_seq,  maxlen=max_length, padding='post', truncating='post')

print('Vocabulary size      :', total_words)
print('Max tweet length     :', max_length)
print('Encoded X Train shape:', x_train_pad.shape)
print('Encoded X Test shape :', x_test_pad.shape)
print()
print('Encoded X Train (first 3 rows):')
print(x_train_pad[:3])

## Step 4: Build Model 1 — LSTM

In [ ]:
# Hyperparameters
EMBED_DIM = 64
LSTM_OUT  = 128
DROPOUT   = 0.3
EPOCHS    = 10
BATCH_SIZE = 64

# LSTM Architecture
lstm_model = Sequential(name='LSTM_Model')
lstm_model.add(Embedding(total_words, EMBED_DIM, input_length=max_length))
lstm_model.add(Dropout(DROPOUT))
lstm_model.add(LSTM(LSTM_OUT, return_sequences=False))
lstm_model.add(Dropout(DROPOUT))
lstm_model.add(Dense(64, activation='relu'))
lstm_model.add(Dense(1,  activation='sigmoid'))

lstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print(lstm_model.summary())

## Step 5: Train LSTM Model

In [ ]:
import os
os.makedirs('models', exist_ok=True)

lstm_checkpoint = ModelCheckpoint(
    'models/LSTM_PHM.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

lstm_history = lstm_model.fit(
    x_train_pad, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.1,
    callbacks=[lstm_checkpoint, early_stop],
    verbose=1
)

## Step 6: Build Model 2 — Bidirectional LSTM (Bi-LSTM)

In [ ]:
# Bi-LSTM Architecture
bilstm_model = Sequential(name='BiLSTM_Model')
bilstm_model.add(Embedding(total_words, EMBED_DIM, input_length=max_length))
bilstm_model.add(Dropout(DROPOUT))
bilstm_model.add(Bidirectional(LSTM(LSTM_OUT, return_sequences=False)))
bilstm_model.add(Dropout(DROPOUT))
bilstm_model.add(Dense(64, activation='relu'))
bilstm_model.add(Dense(1,  activation='sigmoid'))

bilstm_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
print(bilstm_model.summary())

## Step 7: Train Bi-LSTM Model

In [ ]:
bilstm_checkpoint = ModelCheckpoint(
    'models/BiLSTM_PHM.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

early_stop_bi = EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True,
    verbose=1
)

bilstm_history = bilstm_model.fit(
    x_train_pad, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.1,
    callbacks=[bilstm_checkpoint, early_stop_bi],
    verbose=1
)

## Step 8: Evaluate Both Models on Test Set

In [ ]:
# --- LSTM Evaluation ---
lstm_pred_probs = lstm_model.predict(x_test_pad)
lstm_pred       = (lstm_pred_probs >= 0.5).astype(int)

lstm_correct = sum(1 for i, y in enumerate(y_test) if y == lstm_pred[i])
lstm_acc     = lstm_correct / len(lstm_pred) * 100

print('=== LSTM Model Test Results ===')
print(f'Correct Predictions : {lstm_correct}')
print(f'Wrong Predictions   : {len(lstm_pred) - lstm_correct}')
print(f'Test Accuracy       : {lstm_acc:.2f}%')
print()
print('Classification Report:')
print(classification_report(y_test, lstm_pred, target_names=['Non-Personal (0)', 'Personal Health (1)']))

In [ ]:
# --- Bi-LSTM Evaluation ---
bilstm_pred_probs = bilstm_model.predict(x_test_pad)
bilstm_pred       = (bilstm_pred_probs >= 0.5).astype(int)

bilstm_correct = sum(1 for i, y in enumerate(y_test) if y == bilstm_pred[i])
bilstm_acc     = bilstm_correct / len(bilstm_pred) * 100

print('=== Bi-LSTM Model Test Results ===')
print(f'Correct Predictions : {bilstm_correct}')
print(f'Wrong Predictions   : {len(bilstm_pred) - bilstm_correct}')
print(f'Test Accuracy       : {bilstm_acc:.2f}%')
print()
print('Classification Report:')
print(classification_report(y_test, bilstm_pred, target_names=['Non-Personal (0)', 'Personal Health (1)']))

## Step 9: Performance Comparison Table

In [ ]:
# Final performance comparison
lstm_train_acc   = max(lstm_history.history['accuracy'])   * 100
lstm_val_acc     = max(lstm_history.history['val_accuracy'])* 100
lstm_train_loss  = min(lstm_history.history['loss'])
lstm_val_loss    = min(lstm_history.history['val_loss'])

bilstm_train_acc  = max(bilstm_history.history['accuracy'])    * 100
bilstm_val_acc    = max(bilstm_history.history['val_accuracy']) * 100
bilstm_train_loss = min(bilstm_history.history['loss'])
bilstm_val_loss   = min(bilstm_history.history['val_loss'])

comparison = pd.DataFrame({
    'Metric'            : ['Best Training Accuracy (%)', 'Best Validation Accuracy (%)', 'Test Accuracy (%)', 'Best Training Loss', 'Best Validation Loss'],
    'LSTM'              : [f'{lstm_train_acc:.2f}',   f'{lstm_val_acc:.2f}',   f'{lstm_acc:.2f}',   f'{lstm_train_loss:.4f}',   f'{lstm_val_loss:.4f}'],
    'Bi-LSTM'           : [f'{bilstm_train_acc:.2f}', f'{bilstm_val_acc:.2f}', f'{bilstm_acc:.2f}', f'{bilstm_train_loss:.4f}', f'{bilstm_val_loss:.4f}']
})

print('=== Model Performance Comparison ===')
print(comparison.to_string(index=False))

## Step 10: Accuracy and Loss Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('LSTM vs Bi-LSTM: Training Performance', fontsize=15, fontweight='bold')

# LSTM Accuracy
axes[0, 0].plot(lstm_history.history['accuracy'],     label='Train Accuracy', color='royalblue',   linewidth=2)
axes[0, 0].plot(lstm_history.history['val_accuracy'], label='Val Accuracy',   color='darkorange',  linewidth=2)
axes[0, 0].set_title('LSTM - Accuracy')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# LSTM Loss
axes[0, 1].plot(lstm_history.history['loss'],     label='Train Loss', color='royalblue',  linewidth=2)
axes[0, 1].plot(lstm_history.history['val_loss'], label='Val Loss',   color='darkorange', linewidth=2)
axes[0, 1].set_title('LSTM - Loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Bi-LSTM Accuracy
axes[1, 0].plot(bilstm_history.history['accuracy'],     label='Train Accuracy', color='seagreen',   linewidth=2)
axes[1, 0].plot(bilstm_history.history['val_accuracy'], label='Val Accuracy',   color='crimson',    linewidth=2)
axes[1, 0].set_title('Bi-LSTM - Accuracy')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Bi-LSTM Loss
axes[1, 1].plot(bilstm_history.history['loss'],     label='Train Loss', color='seagreen',  linewidth=2)
axes[1, 1].plot(bilstm_history.history['val_loss'], label='Val Loss',   color='crimson',   linewidth=2)
axes[1, 1].set_title('Bi-LSTM - Loss')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lstm_bilstm_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved as lstm_bilstm_performance.png')

## Step 11: Discussion

### Model Architecture
Both models share the same hyperparameter configuration for a fair comparison:
- **Embedding Dimension:** 64
- **LSTM Units:** 128
- **Dropout Rate:** 0.3 (applied after Embedding and after LSTM layer)
- **Dense Layer:** 64 units (ReLU) → 1 unit (Sigmoid for binary classification)
- **Optimizer:** Adam | **Loss:** Binary Crossentropy
- **Callbacks:** EarlyStopping (patience=3), ModelCheckpoint (best val_accuracy)

### LSTM Model
The standard LSTM processes the tweet sequence in a **single forward direction**, capturing temporal dependencies from left to right. It achieves strong performance on short tweet text where context generally flows forward.

### Bi-LSTM Model
The Bi-LSTM processes the sequence in **both forward and backward directions**, effectively doubling the contextual information available at each timestep. This is particularly beneficial for tweets where key health-related words may appear at the end of the sentence and influence earlier context.

### Key Observations
- The **Bi-LSTM** is expected to outperform the standard LSTM due to richer bidirectional context — especially helpful for health mention detection where symptom keywords can appear anywhere in the tweet.
- **Dropout layers** were added to prevent overfitting given the informal, noisy nature of Twitter data.
- **EarlyStopping** ensures training halts before overfitting, and the best model weights are restored automatically.
- The dataset is moderately imbalanced (more non-personal mentions than personal), which is reflected in the precision/recall per class in the classification report above.